## 0 · Setup — clone repo, install deps

**Environment:** you're driving a **Colab GPU runtime from VS Code**. Code runs on the Colab VM; files land under `/content/HE-IFD` — view/download results from the **VS Code remote Explorer** (right-click ▸ Download). Make sure the runtime is **GPU** (T4 is fine). The repo is public (no token).

In [ ]:
import os
if not os.path.isdir("/content/HE-IFD"):
    !git clone -q https://github.com/hkanpak21/HE-IFD.git /content/HE-IFD
%cd /content/HE-IFD
!git pull -q origin master
# torch/torchvision are preinstalled on Colab; add the rest:
!pip -q install transformers datasets timm
!git log --oneline -1

In [ ]:
import torch
ok = torch.cuda.is_available()
print("CUDA:", ok, "|", torch.cuda.get_device_name(0) if ok else "NO GPU — Runtime ▸ Change runtime type ▸ T4 GPU")

# 028 · MIA on pretrained backbones — ViT/CIFAR-100 + RoBERTa/AG-News

Extends the 021 `mia/` suite to a pretrained backbone in **both** modalities. 3 attacks (Yeom/LiRA/GLiRA) × 3 surfaces (external/fellow/prototype), ~64 shadows. Mirrors `jobs/heifd_021_mia_vit_cifar100.sh` + `jobs/heifd_028_mia_roberta_agnews.sh`.

**Heaviest notebook** (64 shadows × 2 backbones). Strongly consider the **Drive persistence** cell so a VS Code/Colab disconnect resumes from `shadows/<cell>/` checkpoints. Smoke-test with `N_SHADOWS = 8` first, then 64.

**Gate:** report released-model AUC + prototype-channel DP-collapse per backbone; flag any deviation from the MNIST dual story.

In [ ]:
# OPTIONAL — persist cache/ + results/ to Google Drive so a dropped Colab
# session resumes (recommended for 028; the MIA suite resumes from
# shadows/<cell>/ checkpoints). In the VS Code frontend drive.mount uses the
# auth-code paste flow. Leave commented to keep everything on the Colab VM.
#
# from google.colab import drive
# drive.mount("/content/drive")
# BASE = "/content/drive/MyDrive/heifd_colab"
# os.makedirs(BASE + "/cache", exist_ok=True); os.makedirs(BASE + "/results", exist_ok=True)
# CACHE_ROOT = BASE + "/cache"; RESULTS_ROOT = BASE + "/results"; print("persisting to", BASE)

In [ ]:
# Output roots. If you mounted Drive above it already set CACHE_ROOT/RESULTS_ROOT;
# otherwise these local (Colab VM) defaults apply.
import os
DATA_ROOT = "data"
CACHE_ROOT = globals().get("CACHE_ROOT", "cache")
RESULTS_ROOT = globals().get("RESULTS_ROOT", "results")
print("DATA_ROOT=%s  CACHE_ROOT=%s  RESULTS_ROOT=%s" % (DATA_ROOT, CACHE_ROOT, RESULTS_ROOT))

In [ ]:
import torchvision as tv
print("downloading datasets into", DATA_ROOT, "...")
tv.datasets.CIFAR100(DATA_ROOT, train=True, download=True)
tv.datasets.CIFAR100(DATA_ROOT, train=False, download=True)
print("vision datasets ready")

In [ ]:
# AG-News (HF dataset) + RoBERTa weights for the language cell.
from datasets import load_dataset; load_dataset("ag_news")
from transformers import AutoTokenizer, AutoModel
AutoTokenizer.from_pretrained("roberta-base"); AutoModel.from_pretrained("roberta-base")
print("ag_news + roberta-base ready")

In [ ]:
N_SHADOWS = 64   # set 8 for a quick smoke run first
POOL = 5000

### Run A — ViT / CIFAR-100 (vision)

In [ ]:
!python -m mia.run \
    --backbones vit_b32_cifar100 \
    --Ns 10 --alphas 0.05,1.0 --methods raw_union_K20 --seeds 42 \
    --n-shadows $N_SHADOWS --attack-pool-size $POOL --prototype-K-per-class 20 \
    --case heifd_021_mia \
    --data-root $DATA_ROOT --cache-root $CACHE_ROOT --results-root $RESULTS_ROOT

### Run B — RoBERTa / AG-News (language)

In [ ]:
!python -m mia.run \
    --backbones roberta_base_agnews \
    --Ns 10 --alphas 0.05,1.0 --methods raw_union_K20 --seeds 42 \
    --n-shadows $N_SHADOWS --attack-pool-size $POOL --prototype-K-per-class 20 \
    --case heifd_021_mia \
    --data-root $DATA_ROOT --cache-root $CACHE_ROOT --results-root $RESULTS_ROOT

### Results — released-model AUC + prototype DP-collapse, both backbones

In [ ]:
from pathlib import Path
rm = Path(RESULTS_ROOT) / "heifd_021_mia" / "README.md"
print(rm.read_text() if rm.exists() else "(no README yet)")

In [ ]:
import json, glob
for p in sorted(glob.glob(f"{RESULTS_ROOT}/heifd_021_mia/*summary*.json")):
    print("==", p, "=="); print(json.dumps(json.load(open(p)), indent=2)[:4000])

In [ ]:
# Bundle results/heifd_021_mia/ for retrieval. In VS Code you can instead just
# right-click results/heifd_021_mia/ in the remote Explorer and Download.
import shutil
out = shutil.make_archive("/content/heifd_021_mia_results", "zip", f"{RESULTS_ROOT}/heifd_021_mia")
print("zipped ->", out, "\nDownload via the VS Code Explorer (right-click ▸ Download).")